# MTAM Reproduction — ZuCo Sentiment Analysis (sentence-level)

Runs the **forked** repo `parmisbathaeiyan/EEG_Language_Alignment`, branch `reproduction`,
instead of patching upstream code inside this notebook. Every change lives as a git
commit on the fork, so `git log upstream/main..reproduction` is the exact list of
deviations from the published code (printed in cell 2).

Uses the **original ZuCo `.mat` data** with the upstream `prepare_sr_eeg_data` loader —
no custom dataloader. The sentiment-label CSV ships in the repo
(`preprocessed/ZuCo/sentiment_labels_clean.csv`), so only the `.mat` files come from Drive.

**Branch state (commits over upstream):**
- compat (modern PyTorch/Colab) + reporting (labeled confusion matrix, macro P/R/F1, JSON/PNG)
- bug fixes: best-val checkpoint (was last-epoch); test loader keeps the final partial batch;
  all RNGs seeded (`--seed`); configurable patience / early-stop delta; eeg_dict disk cache
- **methodology fixes so far:** 80/10/10 split (was 60/10/30); `--oversample` implemented
  (off by default); **standard scaled-dot-product attention** — softmax over keys, no
  in-attention BatchNorm (paper App B.2) ← latest

**Still pending (next commits):** turn on oversampling, BERT last-4-layer averaging,
warmup/LR retune, MLP/ResNet baselines, single-modality (Ours-EEG/Text) heads.

Data stays on Drive / in the repo; `data/` is gitignored and never committed.

## 1. Mount Drive & configure paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── EDIT THIS to point at your original ZuCo task-1 (SR) .mat files ────────────
# Folder should contain results<SUBJ>_SR.mat (e.g. resultsZAB_SR.mat, ...).
OG_ZUCO_SR_DIR = '/content/drive/MyDrive/Parmis/Thesis/Data/zuco_og_raw'
RESULTS_ROOT   = '/content/drive/MyDrive/Parmis/Thesis/Results/reproduce_EEG_Language_Alignment'
EEG_CACHE      = '/content/eeg_dict_cache.pkl'   # caches the processed .mat data between
                                                 # runs (lives outside the repo, survives
                                                 # re-clone). Delete it if you change data.

FORK_URL = 'https://github.com/parmisbathaeiyan/EEG_Language_Alignment.git'
BRANCH   = 'reproduction'
# ──────────────────────────────────────────────────────────────────────────────

import os
assert os.path.isdir(OG_ZUCO_SR_DIR), f'SR .mat folder not found: {OG_ZUCO_SR_DIR}'
mats = sorted(f for f in os.listdir(OG_ZUCO_SR_DIR) if f.endswith('.mat'))
assert mats, f'No .mat files in {OG_ZUCO_SR_DIR}'
os.makedirs(RESULTS_ROOT, exist_ok=True)
print('Drive mounted. SR .mat files:', mats)

## 2. Clone the fork (reproduction branch)

In [ ]:
%cd /content
!rm -rf /content/EEG_Language_Alignment
!git clone --branch {BRANCH} {FORK_URL}
%cd /content/EEG_Language_Alignment
print('\nDeviations from upstream:')
!git remote add upstream https://github.com/Jason-Qiu/EEG_Language_Alignment.git 2>/dev/null; git fetch -q upstream
!git log --oneline upstream/main..{BRANCH}

## 3. Install dependencies
The repo's `requirements.txt` is frozen for Python 3.7 / CUDA 11.3. Install modern,
Colab-compatible versions instead.

In [ ]:
!pip install -q transformers==4.40.0
!pip install -q POT           # Python Optimal Transport (Wasserstein loss path)
!pip install -q scipy numpy pandas scikit-learn tqdm matplotlib seaborn
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 4. Prepare data
The upstream loader reads `data/SR/*.mat` and `data/sentiment_labels_clean.csv`.
Symlink the `.mat` files into `data/SR/`, copy the repo's label CSV, make output dirs.

In [ ]:
%cd /content/EEG_Language_Alignment
import os, shutil

os.makedirs('data/SR', exist_ok=True)
for d in ['lr_curves', 'pred_labels', 'baselines']:
    os.makedirs(d, exist_ok=True)

# Symlink the original .mat files into data/SR/ (no copy).
for f in os.listdir(OG_ZUCO_SR_DIR):
    if f.endswith('.mat'):
        dst = f'data/SR/{f}'
        if not os.path.exists(dst):
            os.symlink(os.path.join(OG_ZUCO_SR_DIR, f), dst)

# Label CSV ships in the repo.
shutil.copy('preprocessed/ZuCo/sentiment_labels_clean.csv', 'data/sentiment_labels_clean.csv')

print('data/SR:', sorted(os.listdir('data/SR')))
import pandas as pd
_df = pd.read_csv('data/sentiment_labels_clean.csv')
print('labels:', _df.shape, '| dist:', _df['sentiment_label'].value_counts().to_dict())

## 5. Smoke test (dev mode)
One sample, 2 epochs — catches path/import errors before a full run.

In [ ]:
!python main_new.py --dataset ZuCo --task SA --level sentence \
    --modality fusion --model transformer \
    --batch_size 8 --inference 0 --dev 1 --loss CE \
    --epochs 2 --device cuda --num_layers 2 --num_heads 5

## 6. Run harness
Logs stream to screen and to a `.txt`; a results `.json` and learning-curve `.png`
are written to Drive under `RESULTS_ROOT/<folder>/`.

In [ ]:
import subprocess, os, datetime

def run_experiment(modality, loss, folder_name, *,
                   model='transformer', level='sentence',
                   num_layers=1, num_heads=5, batch_size=64,
                   epochs=200, warm_steps=2000, dropout=0.3,
                   patience=20, oversample=1, suffix=''):
    ts       = datetime.datetime.now().strftime('%Y%m%d%H%M%S')
    run_name = f'{model}_{modality}_{level}_{loss}_L{num_layers}H{num_heads}b{batch_size}_ws{warm_steps}_p{patience}' + (f'_{suffix}' if suffix else '')
    run_dir  = os.path.join(RESULTS_ROOT, folder_name)
    os.makedirs(os.path.join(run_dir, 'logs'),  exist_ok=True)
    os.makedirs(os.path.join(run_dir, 'plots'), exist_ok=True)
    log_path  = os.path.join(run_dir, 'logs',  f'{run_name}_{ts}.txt')
    json_path = os.path.join(run_dir,          f'{run_name}_{ts}.json')
    plot_dst  = os.path.join(run_dir, 'plots', f'{run_name}_{ts}.png')

    cmd = ['python', '-u', 'main_new.py',
           '--dataset', 'ZuCo', '--task', 'SA', '--level', level,
           '--modality', modality, '--model', model, '--loss', loss,
           '--batch_size', str(batch_size), '--epochs', str(epochs),
           '--num_layers', str(num_layers), '--num_heads', str(num_heads),
           '--dropout', str(dropout), '--warm_steps', str(warm_steps),
           '--patience', str(patience), '--oversample', str(oversample),
           '--eeg_cache', EEG_CACHE,
           '--inference', '0', '--dev', '0', '--device', 'cuda',
           '--timestamp', ts, '--json_path', json_path, '--plot_dst', plot_dst]

    print(f'Run: {run_name} | {ts}\nDir: {run_dir}\n' + '-'*60)
    with open(log_path, 'w') as lf:
        lf.write(f'Run: {run_name}\nTimestamp: {ts}\nArgs: {" ".join(cmd[1:])}\n' + '-'*60 + '\n')
        p = subprocess.Popen(cmd, cwd='/content/EEG_Language_Alignment',
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1,
                             env={**os.environ, 'TQDM_DISABLE': '1',
                                  'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'})
        for line in p.stdout:
            print(line, end=''); lf.write(line); lf.flush()
        p.wait()
    print('-'*60 + f'\nDone. Log -> {log_path}')

print('run_experiment() ready.')

## 7. Beep (run when you want a sound)
Run this cell manually whenever you want an audible beep — e.g. after a long run
finishes. Plays in the Colab tab; no setup, no popups.

In [ ]:
import numpy as np
from IPython.display import Audio, display
dur = 3.0   # seconds (longer = bigger); 880 = pitch in Hz; 0.3 = volume (0-1)
fr = 44100
t = np.linspace(0, dur, int(fr*dur), endpoint=False)
display(Audio(0.3*np.sin(2*np.pi*880*t), rate=fr, autoplay=True))

## 8. Current test — Fix #1: standard scaled-dot-product attention

The latest commit (`restore standard scaled dot-product attention`) makes the EEG/text
encoders' attention match the paper (App B.2): softmax over **keys** (`dim=-1`, was `dim=0`
= over the batch), no undocumented BatchNorm over the score matrix, proper `-1e9` masking.

The cell below re-runs the **exact reference config** from the previous run (v6: 2 layers /
5 heads / batch 64 / warmup 2000 / patience 20, fusion CCA+WD, **oversample off**, which
scored test acc **0.605**). The only difference now is the attention fix, so any change is
attributable to it. Compare against 0.605 (prior) and the paper's 0.826.

The second (optional) cell re-runs the **4-layer / batch-32** config, which *always*
collapsed to the majority class before — the key thing to check is whether the attention
fix stops that collapse.

In [ ]:
# === Fix #1 test: standard scaled-dot-product attention (commit "restore standard ... attention") ===
# Same config as the previous reference run (v6: 2 layers / 5 heads / batch 64 / ws2000 / patience 20,
# fusion CCAWD, oversample OFF), so the ONLY change vs v6 (test acc 0.605) is the attention fix.
# Clean A/B: did fixing softmax(dim=-1) + dropping the in-attention BatchNorm change anything?
# Keep oversample=0 here — turning it on is the NEXT step, kept separate for attribution.
run_experiment(modality='fusion', loss='CCAWD', folder_name='fusion_CCAWD_attnFix',
               num_layers=2, num_heads=5, batch_size=64, warm_steps=2000,
               patience=20, oversample=0)

In [ ]:
# === (optional) collapse test: 4 layers / 4 heads / batch 32 ===
# This config collapsed to the majority class on every prior run (cm [[0,42,0],[0,43,0],[0,38,0]]).
# If the attention fix is the real cause, this should now actually train. oversample still OFF.
run_experiment(modality='fusion', loss='CCAWD', folder_name='fusion_CCAWD_attnFix',
               num_layers=4, num_heads=4, batch_size=32, warm_steps=2000,
               patience=20, oversample=0)

---
### Next on the `reproduction` branch
Paper-vs-code discrepancies, one commit each, re-run after each:
seeded split → 80/10/10 split → per-batch oversampling → BERT last-4-layer averaging
→ warmup/LR schedule. Then wire the MLP / ResNet baselines and single-modality runs.
To pick up new commits: re-run cell 2 (it re-clones the branch cleanly).